# LAPORAN PRAKTIKUM APLIKASI WEBSITE (MODUL 3 & 4)
## Tahap 1: Preprocessing dan Exploratory Data Analysis (EDA) Data Finansial Perusahaan

**Deskripsi Tugas:**
Mencari dataset mandiri, melakukan pra-pemrosesan data (*data preprocessing*), eksplorasi data (*EDA*), dan membangun aplikasi web visualisasi interaktif menggunakan Streamlit dengan menerapkan konsep *caching*, *progress bar*, *widgets*, dan *session state/callback* yang telah dipelajari pada Modul 3 dan Modul 4.

**Dataset:** `company_data.csv`  
Dataset ini berisi indikator performa keuangan dan pasar dari 1.000 perusahaan, meliputi:
- `company_id`: ID unik perusahaan
- `price`: Harga saham saat ini
- `volume`: Volume perdagangan saham harian
- `pe_ratio`: Rasio harga terhadap laba (*Price to Earnings Ratio*)
- `revenue`: Total pendapatan tahunan perusahaan
- `profit`: Total laba bersih tahunan perusahaan
- `employees`: Jumlah total karyawan

### 1. Mengimpor Library yang Dibutuhkan
Pada tahap ini kita memuat pustaka utama Python untuk pengolahan data (`pandas`, `numpy`) serta pustaka visualisasi grafik (`matplotlib`, `seaborn`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Konfigurasi gaya tampilan grafik
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("Semua library berhasil dimuat!")

### 2. Memuat Dataset Mentah (*Raw Data*)
Membaca file `company_data.csv` dan melihat sampel data teratas.

In [ ]:
df_raw = pd.read_csv('company_data.csv')
print(f"Dimensi Dataset: {df_raw.shape[0]} baris dan {df_raw.shape[1]} kolom.")
df_raw.head(10)

### 3. Pemeriksaan Struktur dan Kualitas Data
Mengecek tipe data setiap kolom, ketiadaan nilai (*missing values / null*), dan baris duplikat.

In [ ]:
print("=== INFORMASI DATASET ===")
df_raw.info()

print("\n=== JUMLAH MISSING VALUES PER KOLOM ===")
print(df_raw.isnull().sum())

print("\n=== JUMLAH BARIS DUPLIKAT ===")
print(f"Duplikat: {df_raw.duplicated().sum()}")

### 4. Pembersihan Data (*Data Cleaning*)
Langkah pembersihan yang dilakukan:
1. Memastikan standarisasi nama kolom menggunakan huruf kecil (*lowercase*) dan tanpa spasi liar, sebagaimana dicontohkan pada Modul 4.
2. Membulatkan angka pecahan (*floating points*) ke 2 angka di belakang koma untuk kemudahan interpretasi dan keterbacaan.

In [ ]:
df = df_raw.copy()

# Standarisasi penamaan kolom (lowercase)
df.columns = [col.strip().lower() for col in df.columns]

# Pembulatan nilai float agar rapi
float_cols = ['price', 'pe_ratio', 'revenue', 'profit']
df[float_cols] = df[float_cols].round(2)

df.head()

### 5. Rekayasa Fitur (*Feature Engineering*)
Untuk menghasilkan visualisasi yang mendalam di aplikasi web Streamlit, kita menambahkan beberapa variabel bisnis penting:
1. **`profit_margin` (%)**: Mengukur rasio efisiensi keuntungan terhadap pendapatan: `(profit / revenue) * 100`.
2. **`rev_per_employee`**: Pendapatan per karyawan untuk mengukur produktivitas: `revenue / employees`.
3. **`valuation_status`**: Kategori valuasi berdasarkan Rasio P/E:
   - `< 15`: *Undervalued* (Harga saham dinilai murah/berpotensi *bargain*)
   - `15 - 30`: *Fair Value* (Valuasi wajar)
   - `> 30`: *Overvalued* (Valuasi tinggi / saham bertumbuh/mahal)
4. **`company_size`**: Skala ukuran perusahaan berdasarkan jumlah karyawan (*Small*, *Medium*, *Large*, *Enterprise*).

In [ ]:
# Menghitung Profit Margin dan Revenue per Employee
df['profit_margin'] = ((df['profit'] / df['revenue']) * 100).round(2)
df['rev_per_employee'] = (df['revenue'] / df['employees']).round(4)

# Klasifikasi Valuasi P/E
def classify_valuation(pe):
    if pe < 15:
        return 'Undervalued'
    elif pe <= 30:
        return 'Fair Value'
    else:
        return 'Overvalued'

# Klasifikasi Skala Perusahaan
def classify_size(emp):
    if emp < 50000:
        return 'Small (<50k)'
    elif emp <= 100000:
        return 'Medium (50k-100k)'
    elif emp <= 150000:
        return 'Large (100k-150k)'
    else:
        return 'Enterprise (>150k)'

df['valuation_status'] = df['pe_ratio'].apply(classify_valuation)
df['company_size'] = df['employees'].apply(classify_size)

print("Distribusi Status Valuasi:")
print(df['valuation_status'].value_counts())
print("\nDistribusi Ukuran Perusahaan:")
print(df['company_size'].value_counts())
df.head()

### 6. Statistik Deskriptif Dataset
Melihat ringkasan statistik (rata-rata, standar deviasi, kuartil, nilai min dan max) dari semua variabel numerik.

In [ ]:
df.describe().T

### 7. Exploratory Data Analysis (EDA) dan Visualisasi Data
Kita memvisualisasikan data ke dalam beberapa plot untuk melihat korelasi dan distribusi.

In [ ]:
# Visualisasi 1: Distribusi Harga Saham dan Rasio P/E
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['price'], kde=True, ax=axes[0], color='royalblue', bins=25)
axes[0].set_title('Distribusi Harga Saham (Price)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Stock Price ($)')
axes[0].set_ylabel('Frekuensi')

sns.histplot(df['pe_ratio'], kde=True, ax=axes[1], color='crimson', bins=25)
axes[1].set_title('Distribusi P/E Ratio', fontsize=13, fontweight='bold')
axes[1].set_xlabel('P/E Ratio')
axes[1].set_ylabel('Frekuensi')

plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi 2: Hubungan Pendapatan (Revenue) vs Laba (Profit) berdasarkan Valuasi
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x='revenue',
    y='profit',
    hue='valuation_status',
    palette={'Undervalued': 'forestgreen', 'Fair Value': 'goldenrod', 'Overvalued': 'firebrick'},
    alpha=0.7,
    s=60
)
plt.title('Hubungan Revenue vs Profit berdasarkan Kategori Valuasi P/E', fontsize=14, fontweight='bold')
plt.xlabel('Revenue ($)')
plt.ylabel('Profit ($)')
plt.legend(title='Status Valuasi')
plt.show()

In [ ]:
# Visualisasi 3: Matriks Korelasi Antar Fitur Numerik
plt.figure(figsize=(9, 7))
numeric_df = df[['price', 'volume', 'pe_ratio', 'revenue', 'profit', 'employees', 'profit_margin']]
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriks Korelasi Indikator Finansial Perusahaan', fontsize=14, fontweight='bold')
plt.show()

### 8. Menyimpan Dataset Bersih (*Cleaned Dataset*)
Dataset yang telah diproses dan diperkaya fitur disimpan ke `company_data_clean.csv` untuk diintegrasikan secara langsung ke dalam aplikasi web Streamlit.

In [ ]:
df.to_csv('company_data_clean.csv', index=False)
print("Dataset bersih berhasil disimpan ke file 'company_data_clean.csv'!")

### 9. Kesimpulan Pra-pemrosesan Data
1. Dataset memiliki 1.000 entri perusahaan tanpa ada *missing values* ataupun baris data duplikat.
2. Berhasil ditambahkan fitur analitik bisnis: `profit_margin`, `rev_per_employee`, `valuation_status`, dan `company_size`.
3. Siap dihubungkan ke dashboard web interaktif Streamlit (`app.py`) yang dilengkapi fitur *caching*, *progress bar*, *interactive sliders/filters*, dan *widgets* interaktif.